# 📘 Introduction to Machine Learning with Scikit-Learn

**AIRCHECK Workshop 2026** · From first principles to a model you could deploy.

This notebook builds up the machine learning workflow end to end: what ML is and
which problems it suits, why `scikit-learn` is the tool of choice, how to train
and evaluate a model properly, what the common algorithms actually do, and what
it takes to move a model out of a notebook and into use.

> **No prior ML experience is assumed.** Comfort with Python, NumPy and Pandas —
> the previous notebook — is enough.

---

# 🔍 Section 1 · What is Machine Learning?

Machine Learning (ML) is a field of computer science that focuses on finding patterns in data. In hyper-simplified non-functional code, the procedure is:

In [ ]:
# --- run-time tracking -----------------------------------------------------
# Times every cell, so the last cell can report how long the whole notebook took.
# Harmless outside Jupyter/Colab, and costs nothing to run.
import time as _time

ECHO_CELL_TIME = False    # True prints each cell's own time under its output

CELL_TIMES = []
_timer_state = {}

try:
    from IPython import get_ipython

    def _timer_pre(info):
        _timer_state["t0"] = _time.perf_counter()
        _timer_state["src"] = getattr(info, "raw_cell", "")

    def _timer_post(result):
        t0 = _timer_state.pop("t0", None)
        if t0 is None:
            return
        src = _timer_state.pop("src", "")
        first = next((l.strip() for l in src.split("\n") if l.strip()), "")
        taken = _time.perf_counter() - t0
        CELL_TIMES.append((taken, first[:70]))
        if ECHO_CELL_TIME:
            print(f"[cell {len(CELL_TIMES):>2}  {taken:6.2f}s]")

    _ip = get_ipython()
    if _ip is not None and not _timer_state.get("registered"):
        _ip.events.register("pre_run_cell", _timer_pre)
        _ip.events.register("post_run_cell", _timer_post)
        _timer_state["registered"] = True
except Exception:
    pass          # timing is a convenience; never let it break the notebook

NOTEBOOK_STARTED = _time.time()
# ---------------------------------------------------------------------------

# Make sure the packages this notebook needs are present.
# Colab and a prepared local environment already have them; some Databricks
# runtimes do not, and a missing package here stops the whole notebook.
#   mlflow is deliberately absent - Section 12 handles it being missing.
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
]

missing = [pkg for module, pkg in REQUIRED if importlib.util.find_spec(module) is None]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("Done.")
else:
    print("All required packages are already available.")

In [ ]:
# patterns = ml_algorithm(data)

There are two popular kinds of problems, also called learning scenarios, that ML practitioners confront.

The simplest learning scenario is called **Unsupervised Learning**. In this case we want to find some hidden structure in the data provided. An archetypal problem in unsupervised learning is **clustering**. Clustering involves taking in a collection $X$ of $n$ vectors/arrays of real numbers $\{x_1, \dots, x_n\}$ where $x_i\in R^m$ and partitioning them into disjoint groups.

In [ ]:
#clusters = clustering_algorithm(X)

The other learning scenario we consider is **supervised learning**. In this case each of the *observation* $x_i$ has a matched **label** $y_i$. The assumption is that there is some function $f$ that connects the observations to the labels, that is
$y_i = f(x_i)$.

The goal of ML in supervised learning is to *learn a model*  $m_f$ that “behaves like $f$.” That is we want $m_f(x_i)$ to be close to $y_i$ for all $i$. Once $m_f$ has been constructed, we can use it to predict the labels on new data $X'$ where we may not know the labels. Again in cartoon code:

In [ ]:
#model = fit_model(X,y)
#predicted_labels = model.predict(X_new)

Supervised learning can be further subdivided into **regression problems** and **classification problems** based on the nature of their labels.

Regression problems have continuous/real-valued labels. An example regression problem is predicting the price of a house from various measurements about it (e.g. size, neighborhood, number of bedrooms, etc.).

For classification problems the labels come from a finite set of categories. An example classification problem is categorizing an e-mail as spam or not spam.

Fortunately we do not need to rewrite ML algorithms from scratch. In python there is a comprehensive library called `scikit-learn` that has implemented many popular and effective machine learning algorithms that can be used "off the shelf."  We will use `scikit-learn` to illustrate the basics of machine learning.

## The learning paradigms

"Machine learning" covers several quite different setups. They are told apart by
**what you hand the algorithm**, not by which algorithm you pick.

| Paradigm | What you give it | What it learns | Example |
|---|---|---|---|
| **Supervised** | inputs `X` **and** labels `y` | a mapping from `X` to `y` | predict whether a compound binds a target |
| **Unsupervised** | inputs `X` only | structure hidden in the data | group compounds into chemical series |
| **Semi-supervised** | a little labelled data, a lot unlabelled | a mapping, using the unlabelled data for extra signal | a screen where only a few hundred compounds were confirmed |
| **Self-supervised** | unlabelled data, with the labels invented from the data itself | general-purpose representations | a language model predicting the next word |
| **Reinforcement** | an environment and a reward signal | a policy: which action to take when | generating molecules that maximise a score |

Most day-to-day work — and everything in this workshop — is **supervised** or
**unsupervised**. The other three matter enormously in research, but they need
either far more data or a simulator to interact with.

## When is machine learning the right tool?

ML earns its keep when a rule is **easy to demonstrate but hard to write down**.
You can show a thousand examples of a hit compound far more easily than you can
write the chemistry rules that make one.

**Reach for ML when:**

- the pattern is real but nobody can state it explicitly
- you have enough labelled examples to cover the cases you care about
- occasional mistakes are tolerable, and you can measure how often they happen

**Do not reach for ML when:**

- a deterministic rule already works — never train a model to check whether a
  molecular weight is under 500
- you have very little data, and no way to get more
- you need a guarantee rather than a probability, or a decision you can defend
  line by line to a regulator

> **A useful reframing**
>
> A model does not "know" chemistry, or medicine, or anything else. It finds
> statistical regularities in the data you gave it. If your data is biased, the
> model reproduces the bias faithfully and confidently.

In [ ]:
# Supervised and unsupervised learning, side by side on the same data

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, adjusted_rand_score

X, y = load_iris(return_X_y=True)

# --- Supervised: we SHOW it the labels -------------------------------
clf = LogisticRegression(max_iter=1000).fit(X, y)
print("Supervised (LogisticRegression)")
print("  we provided labels, and it learned to reproduce them")
print("  accuracy on the data it was trained on: %.3f"
      % accuracy_score(y, clf.predict(X)))

# --- Unsupervised: labels are HIDDEN from the algorithm ---------------
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
print("\nUnsupervised (KMeans)")
print("  it never saw a single label - it only grouped similar flowers")
print("  agreement with the true species (adjusted Rand index): %.3f"
      % adjusted_rand_score(y, km.labels_))

print("\nCluster 0/1/2 are arbitrary names. KMeans found groups that line up")
print("well with the real species, but it could not have told you their names.")

---

# 🗺️ Section 2 · What Kinds of Problems Does ML Solve?

Machine learning is not one technique. It is a family of them, and the useful way
to navigate the family is along two axes: **what kind of answer you need**, and
**what kind of data you have**.

## By the question you are asking

| Task | The question | What comes out | Example in drug discovery |
|---|---|---|---|
| **Classification** | which category? | a label, usually with a probability | is this compound active or inactive? |
| **Regression** | how much? | a number | what is the binding affinity? |
| **Clustering** | what natural groups exist? | a group id per item | which chemical series are in this library? |
| **Dimensionality reduction** | can I compress this? | fewer, denser features | a 2-D UMAP map of a fingerprint space |
| **Ranking** | what order? | a sorted list | which 100 compounds should we test next? |
| **Anomaly detection** | what looks wrong? | an outlier score | plates that failed QC in a screen |

> **Ranking deserves a mention.** In a real screening campaign you rarely need a
> hard active/inactive verdict. You need the top *N* compounds you can afford to
> test. That is a ranking problem, and it is why metrics like enrichment and
> precision-at-K often matter more than accuracy.

## By the kind of data you have

| Data | Typical models | Example |
|---|---|---|
| **Tabular** (rows and columns, feature vectors) | gradient boosting, random forests, linear models | fingerprints to hit prediction |
| **Images** | convolutional networks, vision transformers | cell-painting assays, histopathology slides |
| **Text** | transformer language models | pulling assay results out of papers |
| **Audio / speech** | spectrogram plus CNN or transformer | dictated clinical notes |
| **Time series** | gradient boosting, ARIMA, temporal networks | patient vitals, instrument telemetry |
| **Graphs and molecules** | graph neural networks, or fingerprints plus trees | predicting properties from structure |

> **Where this workshop sits**
>
> The AIRCHECK data is **tabular**. A molecular fingerprint such as `ECFP4` is a
> fixed-length vector of counts, so once the chemistry is encoded, a screening
> hit-prediction model is an ordinary tabular classification problem. That is
> exactly why gradient boosting works so well in the hands-on notebook, and why
> everything you learn here transfers directly to it.
>
> Graph neural networks read the molecular structure itself rather than a
> precomputed fingerprint. They are powerful, but they need far more data and
> compute — fingerprints plus a good tree model remain a very strong baseline.

In [ ]:
# The same library, three different kinds of problem

from sklearn.datasets import load_iris, load_diabetes
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

# --- 1. CLASSIFICATION: which species is this flower? -----------------
X, y = load_iris(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("CLASSIFICATION - predicting a category")
print("  accuracy: %.3f" % accuracy_score(yte, clf.predict(Xte)))
print("  a prediction looks like:", clf.predict(Xte[:5]))

# --- 2. REGRESSION: how far will this disease progress? ---------------
Xd, yd = load_diabetes(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3, random_state=0)
reg = LinearRegression().fit(Xtr, ytr)
print("\nREGRESSION - predicting a number")
print("  mean absolute error: %.1f" % mean_absolute_error(yte, reg.predict(Xte)))
print("  a prediction looks like:", reg.predict(Xte[:3]).round(1))

# --- 3. CLUSTERING: what groups are in here at all? -------------------
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
print("\nCLUSTERING - finding groups with no labels at all")
print("  cluster sizes:", [int((km.labels_ == k).sum()) for k in range(3)])

print("\nNotice that all three used the SAME two method names: .fit() then")
print(".predict(). That consistency is the whole point of scikit-learn.")

---

# 🧰 Section 3 · Why Scikit-Learn?

You could implement logistic regression yourself in an afternoon. You should not
have to, and more importantly you should not have to implement the *fiftieth*
algorithm yourself when you want to compare it against the other forty-nine.

**Scikit-learn** is the standard toolkit for classical machine learning in
Python. Its real contribution is not any single algorithm — it is that every
algorithm is wrapped in the **same interface**.

## The estimator API

Almost everything in scikit-learn is an *estimator*, and estimators expose the
same handful of methods:

| Method | What it does | Who has it |
|---|---|---|
| `.fit(X, y)` | learn from data | every estimator |
| `.predict(X)` | produce predictions | models |
| `.predict_proba(X)` | produce class probabilities | most classifiers |
| `.transform(X)` | change the data | preprocessors such as scalers |
| `.score(X, y)` | a quick default metric | models |

The consequence is that **swapping models is a one-line change**. The code around
the model — splitting, scaling, cross-validation, metrics — never has to move.

## What else you get

- **Pipelines** that chain preprocessing and model into a single object, which is
  the main defence against data leakage
- **Model selection**: `train_test_split`, `KFold`, `GridSearchCV`, `cross_validate`
- **Metrics** for classification, regression, clustering and ranking
- **Preprocessing**: scaling, encoding, imputation, feature selection
- Documentation that is genuinely worth reading, with the maths stated plainly

## When to reach for something else

Scikit-learn is not the answer to everything, and knowing its edges matters:

| Situation | Better tool | Why |
|---|---|---|
| Deep learning on images, text or graphs | **PyTorch**, TensorFlow | scikit-learn has only a basic neural network and no GPU support |
| Large-scale gradient boosting | **LightGBM**, XGBoost | far faster, better with categorical features — this is what the hands-on notebook uses |
| Data too big for memory | **Spark**, Dask, Polars | scikit-learn assumes your data fits in RAM |
| Chemistry-specific featurisation | **RDKit**, DeepChem | fingerprints, descriptors, scaffolds |

> **They cooperate rather than compete.** LightGBM ships a scikit-learn-compatible
> wrapper, so `LGBMClassifier` drops straight into a scikit-learn `Pipeline` and
> `GridSearchCV`. Learning this API pays off well beyond scikit-learn itself.

In [ ]:
# One interface, many models: swapping the estimator is a one-line change

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest":       RandomForestClassifier(random_state=0),
    "SVC":                SVC(probability=True, random_state=0),
}

print("Identical code, three completely different algorithms:\n")
for name, model in models.items():
    model.fit(X_train, y_train)                     # same method
    score = model.score(X_test, y_test)             # same method
    print("  %-20s accuracy %.3f" % (name, score))

# --- A Pipeline behaves exactly like a single estimator ---------------
pipe = Pipeline([
    ("scale", StandardScaler()),                    # has .transform()
    ("clf", LogisticRegression(max_iter=1000)),     # has .predict()
])
pipe.fit(X_train, y_train)

print("\nA Pipeline is itself an estimator:")
print("  accuracy      : %.3f" % pipe.score(X_test, y_test))
print("  predictions   :", pipe.predict(X_test[:5]))
print("  probabilities :", pipe.predict_proba(X_test[:2]).round(3))

print("\nThe scaler is fitted on the training folds ONLY, inside the pipeline.")
print("That is what stops test data leaking into the preprocessing step.")

---

# 📂 Section 4 · Loading & Exploring a Dataset

We are still missing a key ingredient: a dataset. For this we can use one of the many datasets included with `scikit-learn`. In particular we will use the commonly analyzed **Iris** dataset. This dataset contains information about three species of iris flowers. We will, over this and the following sections, build a classifier that inputs information and can classifies them as one of the three species of iris: *setosa*, *versicolor*, and *virginica*.

In addition to `scikit-learn`, which focuses primarily on the machine learning side of things, we need two addition libraries: **NumPy** and **Pandas**. These libraries are used to manage and process data before and after analyzing it with `scikit-learn`. These libraries are installed in the normal way:

In [ ]:
import numpy as np
import pandas as pd


Now we load the dataset proper.

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()

As written the `iris` object is a dictionary with our data and the labels (also called targets) as well as some additional information or *metadata*. As a first step let's pull out the observations $X$ and the labels $y$:

In [ ]:
X, y = iris['data'],iris['target']

Okay, now what do these look like:

In [ ]:
X

In [ ]:
y

This is... not particularly useful. Let's try finding out how large the datasets are. To do this we will use the `.shape` attribute of `NumPy` arrays.

In [ ]:
print(X.shape)
print(y.shape)

So there are 150 observations in our data and $X$ has 4 *features* per observation. It would be good to know what measure each feature (column) of $X$ maps to. We can do this with a simple call.

In [ ]:
iris['feature_names']

It's clear from the code above that $y$ has three categories, encoded as numbers 0, 1, and 2. Finding out which species of iris these correspond to is doable with another line.

In [ ]:
iris['target_names']

It's worth  pulling together a count of how many observations we have for each category.

In [ ]:
label_to_species = {i: iris["target_names"][i] for i in range(len(iris["target_names"]))}
renamed_targets = [label_to_species[j] for j in y]
pd.Series(renamed_targets).value_counts()

As a last step we want to look at features. To do this we make a pandas dataframe (which you can think of as a spreadsheet with some extra bells and whistles) and analyse their properties.

In [ ]:
data = pd.DataFrame(data=X, columns=iris['feature_names'])
print("Mean Value of Each Feature")
print(data.mean(axis=0))
print("\nStandard Deviation of Each Feature")
print(data.std(axis=0))
print("\nCorrelation Between Features")
print(data.corr())

---

# 📐 Section 5 · Splitting Data into Train & Test Sets
As mentioned above supervised learning is designed to predict the response (species) on *new* data. However, we only have the one dataset! In order to evaluate an algorithm on this dataset we randomly divide it into two subsets: a **training** set and a **testing** set. This partition of $X$ is called a *train-test-split*. Usually the training set is much larger than the test set. We set it to be 80% of the data in this example.

`scikit-learn` has built in functionality to perform train-test splits. In order to keep things reproducible we set a random seed so that the train-test split is consistent.

In [ ]:
from sklearn.model_selection import train_test_split

#set the seed
seed = 123456
train_percentage = 0.8
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = train_percentage, random_state=seed)
print(f"The training data has {X_train.shape[0]} observations")
print(f"The testing data has {X_test.shape[0]} observations")

We should probably also check how many of each species are in the training and testing sets...

In [ ]:
training_names = [label_to_species[j] for j in y_train]
testing_names = [label_to_species[j] for j in y_test]

print(f"The counts of species in the training dataset is:\n-----------\n{pd.Series(training_names).value_counts()}\n")
print(f"The counts of species in the testing dataset is:\n-----------\n{pd.Series(testing_names).value_counts()}")

We started with an exactly equal number of each species in the dataset but we have changed this (slightly) by splitting it. This is something to note as we train our model.

---

# ⚙️ Section 6 · Training a Machine Learning Model
Now that we've split our data it's time to train our model. We'll use a simple linear model called **logistic regression** (see the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) for details).

As the names suggest we will use our training data to fit (train) the model and then pass the testing data through it for evaluation.

In [ ]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression()
classifier.fit(X_train,y_train)
predictions = classifier.predict(X_test)
predictions

It is good (and common) practice to *standardize* the data by make it so each feature has mean zero and unit variance. We can use the `StandardScaler` functionality `scikit-learn` has.

In [ ]:
from sklearn.preprocessing import StandardScaler
standardizer = StandardScaler()
# note that the scaler does NOT care about the targets.
standardizer.fit(X_train)


Does standardizing help? Do we do better if we preserve the class distribution when we split the train and test data? In order to answer these questions we need a way to evaluate model performance.

---

# 📊 Section 7 · Evaluating Model Performance
- Using **accuracy, precision, recall, and F1-score** for classification.
- Using **Mean Squared Error (MSE) and R² score** for regression.
- Displaying results using `classification_report()` and `confusion_matrix()`.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
# print(classification_report(y_test, predictions))
print(f"Accuracy is {100*accuracy_score(y_test,predictions)}%")

How nice! We got 100% accuracy. This, however, doesn't tell us very much. Let's try this again but with a smaller percentage of the data used for training. We check across a range of values.

In [ ]:
train_percentages = np.round((10**-1)*np.arange(3,10,0.5),2)
# increase the total number of iterations used to fit the model
classifier = LogisticRegression(max_iter=1000)
for tpct in train_percentages:
  X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = tpct, random_state=seed)
  classifier.fit(X_train,y_train)
  predictions = classifier.predict(X_test)
  print(f"Train Percentage: {np.round(100*tpct,2)} % ---- Accuracy:{np.round(100*accuracy_score(y_test,predictions),2)}%")


So, as one would expect as we increase the amount of data used for model training the better the model does. However, even at low training data we do quite well. Let's try this again but this time with the class distributions preserved. We will use this with the `stratify` argument.

In [ ]:
print("For stratified splits...")
for tpct in train_percentages:
  X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = tpct, random_state=seed,stratify=y)
  classifier.fit(X_train,y_train)
  predictions = classifier.predict(X_test)
  print(f"Train Percentage: {np.round(100*tpct,2)} % ---- Accuracy:{np.round(100*accuracy_score(y_test,predictions),2)}%")


Interesting! If we force the class distribution to be preserved then the model seems to struggle (more). We also see that the relationship between amount of training data and performance is less consistent. What if we try standardizing our data?

In [ ]:
print("For stratified and standardized splits...")
for tpct in train_percentages:
  X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = tpct, random_state=seed,stratify=y)
  scaler = StandardScaler()
  scaler.fit(X_train)
  X_train = scaler.transform(X_train)
  X_test = scaler.transform(X_test)
  classifier.fit(X_train,y_train)
  predictions = classifier.predict(X_test)
  print(f"Train Percentage: {np.round(100*tpct,2)} % ---- Accuracy:{np.round(100*accuracy_score(y_test,predictions),2)}%")


So standardizing doesn't do much. However, this is likely due to the fact that the features are all measured in the same units (cm) and are roughly of the same scale. However, this is not usually true of the datasets we see in practice. In general:

- Your **default** move should be to standardize your data.
- Standardization **must** be done on the train set. If you fit `StandardScaler` on the full data then information will leak and you will over-estimate model performance.

## Accuracy is not enough

Accuracy is the fraction of predictions that were right. On a balanced dataset
like iris it is a fair summary. On an **imbalanced** dataset it is actively
misleading — and screening data is about as imbalanced as data gets. Your
`sample-test.parquet` has 9 actives in 5,000 compounds, which is 0.18%.

A model that predicts "inactive" for absolutely everything scores **99.82%
accuracy** on that file, and is completely worthless.

| Metric | What it asks | When it is the one you want |
|---|---|---|
| **Accuracy** | what fraction did I get right? | balanced classes, errors equally costly |
| **Precision** | of the ones I flagged, how many were real? | flagging is expensive — you will assay every hit |
| **Recall** | of the real ones, how many did I find? | missing one is expensive |
| **F1** | the balance of precision and recall | you need a single number and care about both |
| **ROC-AUC** | how well are the classes separated overall? | roughly balanced classes |
| **PR-AUC** (average precision) | precision across all recall levels | **rare positives — the screening case** |

> **For a screening campaign, prefer PR-AUC.** ROC-AUC can look reassuringly high
> on heavily imbalanced data because the enormous pool of true negatives dominates
> it. Precision-recall focuses on the rare positive class, which is the thing you
> actually care about finding.

## The confusion matrix

Every classification metric is computed from four counts, and looking at them
directly is often more informative than any single number:

|  | Predicted negative | Predicted positive |
|---|---|---|
| **Actually negative** | true negative | false positive (a wasted assay) |
| **Actually positive** | false negative (a missed hit) | true positive |

In [ ]:
# Why accuracy misleads on imbalanced data

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, classification_report)

# A screening-like problem: 2% positives
X, y = make_classification(n_samples=4000, n_features=20, n_informative=5,
                           weights=[0.98, 0.02], random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

print("Positives in the test set: %d of %d (%.1f%%)"
      % (y_test.sum(), len(y_test), 100 * y_test.mean()))

# --- The "predict everything is negative" baseline --------------------
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
real = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_train, y_train)

rows = []
for name, model in [("Always says 'negative'", dummy), ("LogisticRegression", real)]:
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    rows.append((name,
                 accuracy_score(y_test, pred),
                 precision_score(y_test, pred, zero_division=0),
                 recall_score(y_test, pred, zero_division=0),
                 f1_score(y_test, pred, zero_division=0),
                 roc_auc_score(y_test, prob),
                 average_precision_score(y_test, prob)))

print("\n%-24s %9s %10s %8s %8s %9s %8s"
      % ("model", "accuracy", "precision", "recall", "F1", "ROC-AUC", "PR-AUC"))
for r in rows:
    print("%-24s %9.3f %10.3f %8.3f %8.3f %9.3f %8.3f" % r)

print("\nThe useless model wins on ACCURACY and is worthless on everything else.")
print("This is the single most common way to fool yourself with a screening model.")

# --- The confusion matrix behind those numbers ------------------------
cm = confusion_matrix(y_test, real.predict(X_test))
print("\nConfusion matrix for the real model:")
print("                 pred negative  pred positive")
print("  true negative  %13d %14d" % (cm[0, 0], cm[0, 1]))
print("  true positive  %13d %14d" % (cm[1, 0], cm[1, 1]))
print("\n  %d false positives  -> assays run for nothing" % cm[0, 1])
print("  %d false negatives  -> real hits we never tested" % cm[1, 0])

print("\n" + classification_report(y_test, real.predict(X_test),
                                   target_names=["inactive", "active"],
                                   zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13)
ax.set_xticks([0, 1], ["pred inactive", "pred active"])
ax.set_yticks([0, 1], ["true inactive", "true active"])
ax.set_title("Confusion matrix", fontweight="bold")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

---

# 🔁 Section 8 · Cross-Validation in Depth

A single train/test split gives you **one** number. Change the random seed and
you get a different one. On a small dataset the gap between those numbers can be
larger than the difference between the two models you are trying to choose
between — which means a single split can rank your models wrongly.

**Cross-validation** replaces that one number with several, and reports their
spread as well as their average.

## How k-fold works

Split the training data into *k* equal parts, called **folds**. Then, *k* times
over: hold out one fold, fit the model on the other *k*−1, and score it on the
held-out fold. Every row is used for training *k*−1 times and for scoring exactly
once.

```
k = 5, and the fold marked [test] is the one scored that round

round 1   [test] train  train  train  train
round 2    train [test] train  train  train
round 3    train  train [test] train  train
round 4    train  train  train [test] train
round 5    train  train  train  train [test]
```

You end up with five scores. Their **mean** estimates performance; their
**standard deviation** tells you how much to trust that estimate.

## Which splitter to use

| Splitter | Use it when | Why |
|---|---|---|
| `KFold` | regression, or already-balanced classes | plain sequential split of shuffled data |
| `StratifiedKFold` | **classification** — this is the default | keeps each class at the same proportion in every fold |
| `GroupKFold` | rows come in related groups | keeps a whole group on one side of the split |
| `TimeSeriesSplit` | the data is ordered in time | never trains on the future to predict the past |

> **`GroupKFold` matters more than it looks in chemistry.** Molecules from the same
> DEL cycle or the same scaffold are near-duplicates. If some land in training and
> their twins land in testing, your score measures memorisation, not generalisation,
> and it will be far too optimistic. Grouping by scaffold — a *scaffold split* — is
> the rigorous version of this.

## The leakage trap

Preprocessing must be learned on the **training folds only**. If you scale the
whole dataset before cross-validating, the mean and standard deviation used for
scaling already contain information from the held-out fold. The score comes out
too high, and the effect is invisible unless you go looking.

Putting the scaler in a `Pipeline` fixes this permanently: scikit-learn refits the
scaler inside every fold automatically.

In [ ]:
# Cross-validation: why one split is not enough

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     cross_validate, StratifiedKFold, KFold)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)

# ---------------------------------------------------------------------
# 1. A single split is a lottery - here is the spread over 30 seeds
# ---------------------------------------------------------------------
scores = []
for seed in range(30):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3,
                                          random_state=seed, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    scores.append(m.score(Xte, yte))
scores = np.array(scores)

print("Same model, same data, only the split seed changed:")
print("  lowest  accuracy : %.3f" % scores.min())
print("  highest accuracy : %.3f" % scores.max())
print("  spread           : %.3f  <- larger than many real model differences"
      % (scores.max() - scores.min()))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(scores, bins=12, color="#4c72b0", edgecolor="white")
ax.axvline(scores.mean(), color="#c44e52", linestyle="--", linewidth=2,
           label="mean = %.3f" % scores.mean())
ax.set_title("Accuracy from 30 different random splits", fontweight="bold")
ax.set_xlabel("accuracy")
ax.set_ylabel("how often")
ax.legend()
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------
# 2. Cross-validation reports a mean AND a spread
# ---------------------------------------------------------------------
pipe = Pipeline([("scale", StandardScaler()),
                 ("clf", LogisticRegression(max_iter=1000))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
fold_scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")

print("\n5-fold cross-validation")
print("  per-fold scores : ", np.round(fold_scores, 3))
print("  mean +/- std    :  %.3f +/- %.3f" % (fold_scores.mean(), fold_scores.std()))

# ---------------------------------------------------------------------
# 3. Several metrics at once, plus the training scores
# ---------------------------------------------------------------------
results = cross_validate(pipe, X, y, cv=cv,
                         scoring=["accuracy", "f1_macro", "precision_macro"],
                         return_train_score=True)

print("\nSeveral metrics in one pass:")
for key in ["test_accuracy", "test_f1_macro", "test_precision_macro"]:
    print("  %-22s %.3f +/- %.3f"
          % (key.replace("test_", ""), results[key].mean(), results[key].std()))
print("  %-22s %.3f   <- much higher than test would mean overfitting"
      % ("train_accuracy", results["train_accuracy"].mean()))

# ---------------------------------------------------------------------
# 4. Stratified vs plain KFold on class balance
# ---------------------------------------------------------------------
print("\nClass counts in each validation fold:")
for name, splitter in [("KFold          ", KFold(n_splits=5, shuffle=True, random_state=0)),
                       ("StratifiedKFold", StratifiedKFold(n_splits=5, shuffle=True, random_state=0))]:
    counts = [tuple(np.bincount(y[test], minlength=3)) for _, test in splitter.split(X, y)]
    print("  %s %s" % (name, counts))
print("  Stratified keeps every class present in every fold - always use it")
print("  for classification.")

# ---------------------------------------------------------------------
# 5. The leakage trap, measured
# ---------------------------------------------------------------------
leaky = StandardScaler().fit_transform(X)          # fitted on ALL the data - wrong
leaky_score = cross_val_score(LogisticRegression(max_iter=1000),
                              leaky, y, cv=cv).mean()
clean_score = cross_val_score(pipe, X, y, cv=cv).mean()

print("\nScaling before the split (leaky) : %.4f" % leaky_score)
print("Scaling inside a Pipeline (clean): %.4f" % clean_score)
print("On iris the gap is tiny, but it grows with the number of features and")
print("shrinks with the number of rows - exactly the regime screening data is in.")

---

# 🧪 Section 9 · A Tour of Common ML Algorithms

You do not need to know how every algorithm works internally to use it well. You
do need a rough mental model of what each one assumes, because that tells you when
it will fail.

## Supervised algorithms

| Algorithm | The idea in one line | Strengths | Watch out for |
|---|---|---|---|
| **Logistic regression** | draw a straight boundary between classes, in a weighted sum of the features | fast, interpretable coefficients, a strong baseline | can only draw straight boundaries; needs scaling |
| **k-nearest neighbours** | predict whatever the *k* most similar rows are | no training at all; captures odd shapes | slow at prediction; collapses in high dimensions |
| **Decision tree** | ask a series of yes/no questions about single features | readable end to end; no scaling needed | memorises the training data if left unpruned |
| **Random forest** | average hundreds of trees, each on a random slice of the data | robust, hard to misuse, sensible defaults | large models; less interpretable than one tree |
| **Gradient boosting** | add trees one at a time, each fixing the previous ones' mistakes | usually the best on tabular data | more hyperparameters; will overfit if pushed |
| **Support vector machine** | find the boundary with the widest possible margin; the kernel trick lets that boundary curve | strong on small, high-dimensional data | scales badly past ~10k rows; needs scaling |
| **Naive Bayes** | apply Bayes' rule, pretending features are independent | extremely fast; a good text baseline | that independence assumption is usually false |
| **Neural network (MLP)** | stacked layers of weighted sums and non-linearities | learns arbitrary shapes given enough data | hungry for data and tuning; rarely beats boosting on tabular data |

## Unsupervised algorithms

| Algorithm | The idea in one line | Typical use |
|---|---|---|
| **k-means** | find *k* centres, assign each point to the nearest | quick grouping when you can guess *k* |
| **Hierarchical clustering** | repeatedly merge the two closest groups | when you want a dendrogram rather than a fixed *k* |
| **DBSCAN** | grow clusters through dense regions, leave sparse points unassigned | irregular cluster shapes; finds outliers for free |
| **PCA** | rotate onto the axes with the most variance | compression, de-noising, a fast first look |
| **UMAP / t-SNE** | lay out high-dimensional points in 2-D, keeping neighbours together | visualising a fingerprint space — used in the data-exploration notebook |

> **Where to start on tabular data.** Fit a logistic regression for a baseline, then
> a gradient boosting model. If boosting cannot beat the linear baseline, the
> problem is almost always your features or your labels, not your algorithm.
> Reaching for a neural network at that point will not save you.

## No free lunch

There is a formal result — the *no free lunch* theorem — saying no single algorithm
is best across all possible problems. In practice this means: try several, compare
them with cross-validation, and let the numbers decide.

In [ ]:
# A fair comparison of seven algorithms, using cross-validation

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

X, y = load_iris(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# Each model sits in a Pipeline with a scaler, so the distance- and
# margin-based methods are treated fairly and nothing leaks between folds.
candidates = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "k-nearest neighbours": KNeighborsClassifier(n_neighbors=5),
    "Decision tree": DecisionTreeClassifier(random_state=0),
    "Random forest": RandomForestClassifier(n_estimators=200, random_state=0),
    "Gradient boosting": GradientBoostingClassifier(random_state=0),
    "Support vector machine": SVC(kernel="rbf", random_state=0),
    "Neural network (MLP)": MLPClassifier(hidden_layer_sizes=(32, 16),
                                          max_iter=4000, random_state=0),
    "Naive Bayes": GaussianNB(),
}

results = {}
for name, model in candidates.items():
    pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
    s = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
    results[name] = (s.mean(), s.std())

print("5-fold cross-validated accuracy on the iris dataset\n")
print("%-24s %8s %8s" % ("model", "mean", "std"))
for name, (m, sd) in sorted(results.items(), key=lambda kv: -kv[1][0]):
    print("%-24s %8.3f %8.3f" % (name, m, sd))

# --- Plot the comparison, error bars included -------------------------
order = sorted(results, key=lambda k: results[k][0])
means = [results[k][0] for k in order]
stds = [results[k][1] for k in order]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(order, means, xerr=stds, color="#4c72b0", capsize=4)
ax.set_xlim(0.80, 1.0)
ax.set_xlabel("cross-validated accuracy")
ax.set_title("Seven algorithms, one dataset", fontweight="bold")
plt.tight_layout()
plt.show()

print("The error bars overlap almost completely. On a dataset this small and")
print("this easy, these models are not meaningfully different - and reporting")
print("a winner from a single split would have been noise, not a result.")

---

# 🎯 Section 10 · Hyperparameter Tuning
If you look over the logistic regression documentation you'll see two parameters; a "penalty" parameter and a parameter "C". We are going to try to choose these values (called **hyperparameters**) in order to maximize model performance. In order to do this we will avail ourselves of two useful features that `scikit-learn` has: `Pipeline` and `GridSearchCV`. We will use `Pipeline` to wrap our train-test split protocol and `GridSearchCV` to try all possible combinations of hyperparameters.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# The Pipeline is a named list of steps that get applied in order.
# Since we will be trying multiple penalties we need a solver that
# handles both l1 and l2. 'saga' does, and unlike 'liblinear' it is
# not deprecated for multiclass problems. It is an iterative solver,
# so it needs a generous max_iter to reach convergence.
model_pipe = Pipeline([('scale',StandardScaler()),('clf',LogisticRegression(solver='saga',max_iter=5000))])

# Now we specify the parameters to search over.
# This takes a form of a dictionary where the parameter names are
# taken from the document and we add a prefix so that
# GridSearchCV knows to which step the parameters belong.
# These need to come after two underscores.

param_grid = {
    'clf__C':np.logspace(-4, 4, 20),
    'clf__penalty':['l1','l2']}

In order to choose hyperparameters we first split the data then use what is called stratified k-fold cross validation. In this process the training data is split into k equal subsets, the model is then fit to the remaining k-1 subsets (folds) and evaluated on the held-out fold. The model parameters with the best average performance over all k folds are kept as the best. We choose $k=5$ because our dataset is small. `scikit-learn` uses stratified kfold splits by default for classification problems.

In [ ]:
model = GridSearchCV(model_pipe,param_grid,cv=5)
print("For stratified and standardized splits...")
for tpct in train_percentages:
  X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = tpct, random_state=seed,stratify=y)
  model.fit(X_train,y_train)

  predictions = model.predict(X_test)
  print(f"Train Percentage: {np.round(100*tpct,2)} % ---- Accuracy:{np.round(100*accuracy_score(y_test,predictions),2)}%")



We see that tuning the hyperparameters increases our performance, albeit modestly. Once we have fit the model we can use the best hyper-parameters saved to re-fit on the same model. Let's re-do this with 60% of the data held out. The `GridSearchCV` object has an attribute called `best_estimator_` that stores the model with the best hyperparameters.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size = 0.6, random_state=seed,stratify=y)
model.fit(X_train,y_train)
print(model.best_estimator_)



---

# 💾 Section 11 · Saving & Loading a Trained Model
Once we've built and trained our model we want to save it for future use so that we (or others) won't need to re-train it. There are many [options](https://scikit-learn.org/stable/model_persistence.html) for this in python. We'll use `joblib` for this example.

In [ ]:
import joblib

best_model = model.best_estimator_
joblib.dump(best_model,"./my_model.joblib")
loaded_model = joblib.load("./my_model.joblib")
loaded_model.predict(X_test)

Alright! The model works and can be loaded. The last step, if we want to apply to genuinely new data, is to re-fit the best model on all our data and save it.

In [ ]:
production_model = model.best_estimator_
production_model.fit(X,y)
joblib.dump(production_model,"./production_model.joblib")

---

# 🚀 Section 12 · From Notebook to Production

A model in a notebook helps one person once. Getting value out of it repeatedly
means answering three questions: **which model is this**, **how do I get a
prediction from it**, and **is it still working?**

## Experiment tracking

By the end of a project you will have run hundreds of fits. Without a record you
cannot answer "which settings produced our best result, and on what data?" —
notebook cells get overwritten, and `final_model_v3_REAL.pkl` is not a system.

**MLflow** is the common open-source answer. For each run it records:

- **parameters** — the hyperparameters you used
- **metrics** — the scores you got
- **artifacts** — the model file, plots, the feature list
- **source** — the code version and environment

Then `mlflow ui` gives you a searchable, sortable table of every run.

## The model registry

Tracking captures experiments. A **registry** manages the models you intend to
*use*. It gives each one a name and a version, records a stage — `Staging`,
`Production`, `Archived` — and keeps who promoted what and when. Rolling back
becomes "point at version 4 again" instead of an archaeology expedition.

## Getting predictions out

| Pattern | How it works | Fits when |
|---|---|---|
| **Batch** | a scheduled job scores a file and writes results | screening a library overnight — **this workshop's pattern** |
| **Online API** | a web service answers one request at a time | an interactive tool a chemist clicks in |
| **Embedded** | the model file ships inside another application | an instrument scoring its own readings |

A minimal online service, for reference:

```python
# serve.py  -  run with:  uvicorn serve:app --reload
import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel

model = joblib.load("production_model.joblib")   # loaded once at startup
app = FastAPI()

class Sample(BaseModel):
    features: list[float]

@app.post("/predict")
def predict(sample: Sample):
    x = np.array(sample.features).reshape(1, -1)
    return {
        "prediction": int(model.predict(x)[0]),
        "probability": float(model.predict_proba(x)[0].max()),
    }
```

```bash
curl -X POST http://127.0.0.1:8000/predict \
     -H "Content-Type: application/json" \
     -d '{"features": [5.1, 3.5, 1.4, 0.2]}'
```

## After deployment

A deployed model decays, because the world moves and the model does not.

- **Data drift** — incoming data stops looking like the training data. New chemistry
  in the library is exactly this.
- **Concept drift** — the relationship itself changes; a new assay protocol shifts
  what counts as a hit.
- **Feedback loops** — you only ever assay what the model recommends, so your next
  training set is shaped by the current model's blind spots. This one is easy to
  miss and hard to undo.

Log your predictions, compare feature distributions against training periodically,
and retrain on a schedule rather than waiting for someone to notice.

> **What to take away for the workshop.** You will not stand up MLflow this week.
> But save your model *with its metadata* — the feature list, the training data
> version, the metrics, the date. That single habit is the difference between a
> model someone can pick up in six months and one they have to rebuild.

In [ ]:
# Two habits worth keeping: metadata alongside the model, and run tracking

import json
import platform
from datetime import datetime, timezone

import joblib
import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_iris(return_X_y=True)
iris = load_iris()

pipe = Pipeline([("scale", StandardScaler()),
                 ("clf", LogisticRegression(max_iter=1000, C=1.0))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
pipe.fit(X, y)

# ---------------------------------------------------------------------
# 1. Never save a bare model - save what it needs to be understood later
# ---------------------------------------------------------------------
metadata = {
    "model_name": "iris-species-classifier",
    "version": "1.0.0",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "features": list(iris["feature_names"]),      # order matters at predict time
    "classes": list(iris["target_names"]),
    "n_training_rows": int(X.shape[0]),
    "cv_accuracy_mean": round(float(scores.mean()), 4),
    "cv_accuracy_std": round(float(scores.std()), 4),
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
}

joblib.dump(pipe, "iris_model.joblib")
with open("iris_model.metadata.json", "w") as fh:
    json.dump(metadata, fh, indent=2)

print("Saved iris_model.joblib alongside its metadata:\n")
print(json.dumps(metadata, indent=2))

# --- Reload and confirm the round trip --------------------------------
loaded = joblib.load("iris_model.joblib")
meta = json.load(open("iris_model.metadata.json"))
sample = np.array([[5.1, 3.5, 1.4, 0.2]])          # order must match meta["features"]
pred = loaded.predict(sample)[0]
print("\nReloaded model predicts:", meta["classes"][pred])

# ---------------------------------------------------------------------
# 2. The same run, recorded in MLflow (skipped if it is not installed)
# ---------------------------------------------------------------------
try:
    import mlflow
    import mlflow.sklearn

    mlflow.set_experiment("iris-demo")
    with mlflow.start_run(run_name="logreg-baseline"):
        mlflow.log_params({"model": "LogisticRegression", "C": 1.0, "scaler": "standard"})
        mlflow.log_metrics({"cv_accuracy_mean": float(scores.mean()),
                            "cv_accuracy_std": float(scores.std())})
        mlflow.log_dict(metadata, "metadata.json")
        mlflow.sklearn.log_model(pipe, name="model")
    print("\nLogged one run to MLflow. Browse it with:  mlflow ui")

except ImportError:
    print("\nmlflow is not installed here, so the tracking block was skipped.")
    print("To try it:  pip install mlflow")
    print("Then the code above records params, metrics and the model itself,")
    print("and 'mlflow ui' opens a searchable table of every run you have made.")

---

## ⏱️ How long did that take?

Wall-clock time for the whole notebook, then every cell in the order it ran, and the slowest
five pulled out. Useful for planning a session, and for spotting a cell that is slower than
it looks.

To watch the timings live instead of waiting for this summary, set `ECHO_CELL_TIME = True` in
the first cell — each cell then prints its own time underneath its output.

In [ ]:
elapsed = _time.time() - NOTEBOOK_STARTED
minutes, seconds = divmod(elapsed, 60)

print(f"Total run time : {int(minutes)} min {seconds:04.1f} s")
print(f"Cells executed : {len(CELL_TIMES)}")

if CELL_TIMES:
    measured = sum(t for t, _ in CELL_TIMES)
    print(f"Time in cells  : {measured:.1f} s "
          f"({measured / elapsed:.0%} of the total; the rest is start-up and idle time)")

    # every cell, in the order it ran
    print(f"\n{'cell':>4} {'seconds':>9} {'share':>7}  first line")
    print("-" * 78)
    for n, (taken, first_line) in enumerate(CELL_TIMES, start=1):
        share = taken / measured if measured else 0
        marker = " <-- slow" if taken >= 5 else ""
        print(f"{n:>4} {taken:>9.2f} {share:>6.1%}  {first_line}{marker}")

    print(f"\nSlowest five:")
    for taken, first_line in sorted(CELL_TIMES, reverse=True)[:5]:
        print(f"  {taken:6.1f}s  {first_line}")

print(f"\nMeasured on whatever machine ran this. Colab is usually slower than a laptop,")
print("so treat these as a guide rather than a promise.")
print("Set ECHO_CELL_TIME = True in the first cell to see each cell timed as it runs.")